# Importing libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
import json
import nltk
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words('english'))

# Defining LSTM model

In [2]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers, bidirectional, drop_prob):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers, 
                            bidirectional=bidirectional, batch_first=True, 
                            dropout=drop_prob if n_layers > 1 else 0)
        self.dropout = nn.Dropout(drop_prob)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        
    def forward(self, text):
        embedded = self.dropout(self.embedding(text)) 
        lstm_out, (hidden, cell) = self.lstm(embedded)
        if self.lstm.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
        return self.fc(hidden)

In [3]:
with open('output/vocab.json', 'r') as f:
    vocab = json.load(f)

In [4]:
len(vocab)

20002

In [ ]:
VOCAB_SIZE = len(vocab)
EMBED_DIM = 128
HIDDEN_DIM = 128
OUTPUT_DIM = 3  
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.3
MAX_SEQ_LEN = 50

# Set device (force CPU for inference unless you are doing massive batches)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize the blank model
model = SentimentLSTM(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, BIDIRECTIONAL, DROPOUT)

# Load the trained weights (map_location ensures it works even if trained on GPU and tested on CPU)
model_path = 'output/sentiment_lstm_model.pth' # Or 'best_sentiment_lstm.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)

model.eval() 

SentimentLSTM(
  (embedding): Embedding(20002, 128, padding_idx=0)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=3, bias=True)
)

# Preprocessing input data

In [6]:
def preprocess(text):
    text = re.sub(r'[^a-zA-Z\s]', '', str(text).lower())
    words = [w for w in text.split() if w not in stop_words]
    return words

def encode_text(tokens):
    # Use vocab.get(word, 1) where 1 is the <UNK> (Unknown) token
    encoded = [vocab.get(word, 1) for word in tokens]
    if len(encoded) < MAX_SEQ_LEN:
        encoded += [0] * (MAX_SEQ_LEN - len(encoded)) # 0 is <PAD>
    else:
        encoded = encoded[:MAX_SEQ_LEN]
    return encoded

# Prediction using model

In [ ]:
def predict_custom_text(text):
    # Clean and encode the text
    tokens = preprocess(text)
    encoded = encode_text(tokens)
    
    # Convert to Tensor and add Batch Dimension: shape becomes [1, 50]
    tensor_input = torch.tensor(encoded, dtype=torch.long).unsqueeze(0).to(device)
    
    # Pass through the model without calculating gradients
    with torch.no_grad():
        raw_logits = model(tensor_input)
        
        # Apply Softmax to get percentages
        probabilities = F.softmax(raw_logits, dim=1).squeeze()
        
        # Get the winning index
        winning_index = torch.argmax(probabilities).item()
        confidence = probabilities[winning_index].item() * 100
        
    # Map index back to word
    mapping = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    sentiment = mapping[winning_index]
    
    print(f"Text: '{text}'")
    print(f"Prediction: {sentiment.upper()} (Confidence: {confidence:.2f}%)")
    print(f"Raw Probabilities -> Neg: {probabilities[0]*100:.1f}% | Neu: {probabilities[1]*100:.1f}% | Pos: {probabilities[2]*100:.1f}%\n")
    return sentiment

In [8]:
test_strings = [
    "I absolutely love the new design, it's incredibly fast and easy to use!",
    "My flight was delayed by 6 hours and the staff was extremely rude. Never flying with them again.",
    "I bought a coffee today.",
    "The app is okay, it gets the job done but crashes sometimes.",
    "Completely useless update. You ruined the entire interface."
]

for s in test_strings:
    predict_custom_text(s)

Text: 'I absolutely love the new design, it's incredibly fast and easy to use!'
Prediction: POSITIVE (Confidence: 99.89%)
Raw Probabilities -> Neg: 0.1% | Neu: 0.0% | Pos: 99.9%

Text: 'My flight was delayed by 6 hours and the staff was extremely rude. Never flying with them again.'
Prediction: NEGATIVE (Confidence: 99.74%)
Raw Probabilities -> Neg: 99.7% | Neu: 0.1% | Pos: 0.1%

Text: 'I bought a coffee today.'
Prediction: POSITIVE (Confidence: 87.40%)
Raw Probabilities -> Neg: 3.4% | Neu: 9.2% | Pos: 87.4%

Text: 'The app is okay, it gets the job done but crashes sometimes.'
Prediction: POSITIVE (Confidence: 99.76%)
Raw Probabilities -> Neg: 0.1% | Neu: 0.1% | Pos: 99.8%

Text: 'Completely useless update. You ruined the entire interface.'
Prediction: NEGATIVE (Confidence: 99.00%)
Raw Probabilities -> Neg: 99.0% | Neu: 0.2% | Pos: 0.8%



# Customer inputs

In [9]:
text = 'what you have learned yours and only yours what you want teach different focus the goal not the wrapping paper buddhism can passed others without word about the buddha '

predict_custom_text(text)

Text: 'what you have learned yours and only yours what you want teach different focus the goal not the wrapping paper buddhism can passed others without word about the buddha '
Prediction: NEUTRAL (Confidence: 99.68%)
Raw Probabilities -> Neg: 0.1% | Neu: 99.7% | Pos: 0.3%



'Neutral'

In [12]:
text = 'The movie was good but the ending was bad'

predict_custom_text(text)

Text: 'The movie was good but the ending was bad'
Prediction: POSITIVE (Confidence: 69.22%)
Raw Probabilities -> Neg: 30.2% | Neu: 0.6% | Pos: 69.2%



'Positive'